# FastNN (gRPC) vs SciPy KDTree v0.0.7
Compare KNN results between `fastnn_client.FastNn` and `scipy.spatial.KDTree` on the same source/query data.

In [504]:
import json
from pathlib import Path

import numpy as np
from scipy.spatial import KDTree

from fastnn_client.fast_nn import FastNn
FastNn.DEFAULT_ENDPOINT = "172.31.32.1:50051"  # set explicitly 

# Force local module + explicit endpoint
# import sys
# PYAPP_DIR = Path.cwd()  # assumes notebook is run from PyApp
# if str(PYAPP_DIR) not in sys.path:
#     sys.path.insert(0, str(PYAPP_DIR))

# import fast_nn
# fast_nn.DEFAULT_ENDPOINT = "172.31.32.1:50051"  # set explicitly
# FastNn = fast_nn.FastNn


In [505]:
# Sanity check: tiny dataset
src_tiny = np.asarray([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]], dtype=np.float32)
qry_tiny = np.asarray([[0.1, 0.0, 0.0], [0.9, 0.0, 0.0]], dtype=np.float32)

nn_tiny = FastNn(src_tiny)
dist_tiny, idx_tiny = nn_tiny.query(qry_tiny, k=1)
nn_tiny.close()

print("tiny idx", idx_tiny)
print("tiny dist", dist_tiny)


tiny idx [0 1]
tiny dist [0.1        0.10000002]


In [506]:
# MED 
# data_dir = Path("data/fa79etc/73")
# src_path = data_dir / "set_graph_2998x3.json"
# qry_path = data_dir / "search_locations_6656x3.json"

# baseline_dist_path = data_dir / "distances_6656x10.json"
# baseline_idx_path = data_dir / "nearest_node_indexes_6656x10.json"

#Large
data_dir = Path("data/fa79etc/186")
src_path = data_dir / "set_graph_2998x3.json"
qry_path = data_dir / "search_locations_23920x3.json"

baseline_dist_path = data_dir / "distances_23920x10.json"
baseline_idx_path = data_dir / "nearest_node_indexes_23920x10.json"


src = np.asarray(json.loads(src_path.read_text()))
qry = np.asarray(json.loads(qry_path.read_text()))

src.shape, qry.shape

((2998, 3), (23920, 3))

In [507]:
k = 10

# SciPy KDTree
kdtree = KDTree(src)
sci_dist, sci_idx = kdtree.query(qry, k=k)

# Save SciPy outputs
sci_dist_path = data_dir / "sci_distances.json"
sci_idx_path = data_dir / "sci_indicies.json"
sci_dist_path.write_text(json.dumps(sci_dist.tolist()))
sci_idx_path.write_text(json.dumps(sci_idx.tolist()))

sci_dist.shape, sci_idx.shape

((23920, 10), (23920, 10))

In [508]:
# gRPC FastNN
nn = FastNn(src)
grpc_dist, grpc_idx = nn.query(qry, k=k)
nn.close()

# Save gRPC outputs
grpc_dist_path = data_dir / "grpc_distances.json"
grpc_idx_path = data_dir / "grpc_indicies.json"
grpc_dist_path.write_text(json.dumps(np.asarray(grpc_dist).tolist()))
grpc_idx_path.write_text(json.dumps(np.asarray(grpc_idx).tolist()))

np.asarray(grpc_dist).shape, np.asarray(grpc_idx).shape

((23920, 10), (23920, 10))

In [509]:
def compare_knn(sci_dist, sci_idx, grpc_dist, grpc_idx):
    sci_dist = np.asarray(sci_dist)
    sci_idx = np.asarray(sci_idx)
    grpc_dist = np.asarray(grpc_dist)
    grpc_idx = np.asarray(grpc_idx)

    # Normalize to (Q, K) even when K=1
    if sci_idx.ndim == 1:
        sci_idx = sci_idx.reshape(-1, 1)
    if grpc_idx.ndim == 1:
        grpc_idx = grpc_idx.reshape(-1, 1)
    if sci_dist.ndim == 1:
        sci_dist = sci_dist.reshape(-1, 1)
    if grpc_dist.ndim == 1:
        grpc_dist = grpc_dist.reshape(-1, 1)

    # Ensure shapes are consistent
    if sci_dist.shape != grpc_dist.shape or sci_idx.shape != grpc_idx.shape:
        return {"label": "Unrelated", "reason": "Shape mismatch"}

    # Index overlap per query
    overlap = []
    for i in range(sci_idx.shape[0]):
        a = set(sci_idx[i].tolist())
        b = set(grpc_idx[i].tolist())
        overlap.append(len(a & b) / len(a))
    mean_overlap = float(np.mean(overlap))

    # Distance differences (aligned by rank)
    dist_diff = np.abs(sci_dist - grpc_dist)
    max_diff = float(np.max(dist_diff))
    mean_diff = float(np.mean(dist_diff))

    if mean_overlap > 0.9 and max_diff < 1e-4:
        label = "Very Similar"
    elif mean_overlap > 0.7 and max_diff < 1e-3:
        label = "Similar"
    elif mean_overlap > 0.4:
        label = "Dissimilar"
    else:
        label = "Unrelated"

    return {
        "label": label,
"mean_index_overlap": mean_overlap,
"mean_dist_diff": mean_diff,
"max_dist_diff": max_diff,
}

compare_knn(sci_dist, sci_idx, grpc_dist, grpc_idx)



{'label': 'Very Similar',
 'mean_index_overlap': 0.9999958193979934,
 'mean_dist_diff': 1.5696485498171522e-08,
 'max_dist_diff': 5.980064303641452e-08}

In [510]:
# Compare against baseline results in the same folder
# baseline_dist_path = data_dir / "distances_6656x10.json"
# baseline_idx_path = data_dir / "nearest_node_indexes_6656x10.json"

baseline_dist = np.asarray(json.loads(baseline_dist_path.read_text()))
baseline_idx = np.asarray(json.loads(baseline_idx_path.read_text()))

print("baseline shapes", baseline_dist.shape, baseline_idx.shape)
print("sci vs baseline", compare_knn(sci_dist, sci_idx, baseline_dist, baseline_idx))
print("grpc vs baseline", compare_knn(grpc_dist, grpc_idx, baseline_dist, baseline_idx))


baseline shapes (23920, 10) (23920, 10)
sci vs baseline {'label': 'Very Similar', 'mean_index_overlap': 1.0, 'mean_dist_diff': 0.0, 'max_dist_diff': 0.0}
grpc vs baseline {'label': 'Very Similar', 'mean_index_overlap': 0.9999958193979934, 'mean_dist_diff': 1.5696485498171522e-08, 'max_dist_diff': 5.980064303641452e-08}


In [511]:
sci_idx[0]

array([2682, 2820, 2291, 2507, 2508, 2290, 1656, 1020,  237, 1019])

In [512]:
sci_dist[0]

array([0.00258309, 0.00408228, 0.00575123, 0.00645432, 0.00653706,
       0.00655531, 0.00662647, 0.00695292, 0.00770227, 0.00785808])

In [513]:
grpc_idx[0] 

array([2682, 2820, 2291, 2507, 2508, 2290, 1656, 1020,  237, 1019],
      dtype=int32)

In [514]:
grpc_dist[0]

array([0.00258309, 0.00408227, 0.00575124, 0.0064543 , 0.00653706,
       0.0065553 , 0.00662648, 0.00695292, 0.00770226, 0.00785807],
      dtype=float32)

In [515]:
print("data_dir", data_dir.resolve())
print("src_path", src_path.resolve())
print("qry_path", qry_path.resolve())
print("src[0]", src[0])
print("qry[0]", qry[0])
print("src dtype", src.dtype, "qry dtype", qry.dtype)

data_dir /home/slapd/tbp/data/fa79etc/186
src_path /home/slapd/tbp/data/fa79etc/186/set_graph_2998x3.json
qry_path /home/slapd/tbp/data/fa79etc/186/search_locations_23920x3.json
src[0] [5.55341358e-05 1.49992180e+00 6.60235435e-02]
qry[0] [0.00458398 1.51265432 0.06846508]
src dtype float64 qry dtype float64
